# Imports

In [1]:
from train import train, TrainingConfig
import os
from helpers import load_bpe_tokenization, load_encoding, save_model
import torch
from datasets import StrideDataset
from simpleGPT import SimpleGPTConfig, SimpleGPT
from block import BlockConfig

# Path configuration

In [2]:
models_dir_name = models_dir_path = "models"
model_name = "mickiewicz_gpt"
tokenizers_dir_name = tokenizers_dir_path = "tokenizers"
tokenizer_name = "tokenizer_mickiewicz"
encodings_dir_name = encodings_dir_path = "encodings"
tr_encoding_name = "tr_encoding_mickiewicz"
val_encoding_name = "val_encoding_mickiewicz"

# Tokenizer

In [3]:
encode, decode, vocab, merges = load_bpe_tokenization(os.path.join(tokenizers_dir_path, tokenizer_name + ".pt"))

# Encodings

In [4]:
tr_encoding = load_encoding(os.path.join(encodings_dir_path, tr_encoding_name + ".pt"))
val_encoding = load_encoding(os.path.join(encodings_dir_path, val_encoding_name + ".pt"))

# Datasets

In [5]:
BLOCK_SIZE = 512
tr_dataset = StrideDataset(tr_encoding, BLOCK_SIZE)
val_dataset = StrideDataset(val_encoding, BLOCK_SIZE)

# Model

In [6]:
N_BLOCKS = 16
N_EMBD = 640
N_HEADS = 10
ATTENTION_INNER_DIM = 64
FF_EMBD_TO_DIM_RATIO = 4.0
DROPOUT = 0.1
model = SimpleGPT(SimpleGPTConfig(
    vocab_size=len(vocab),
    n_blocks=N_BLOCKS,
    block_config=BlockConfig(
        n_heads=N_HEADS,
        n_embd=N_EMBD,
        attention_inner_dim=ATTENTION_INNER_DIM,
        ff_embedding_to_dim_ratio=FF_EMBD_TO_DIM_RATIO,
        dropout=DROPOUT
    )
))

In [7]:
print(sum([p.numel() for p in model.parameters()]))

103973638


# Training

In [8]:
BATCH_SIZE = 10
NUM_EPOCHS = 15
LR = 1e-4
INFO_INTERVAL = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
training_config = TrainingConfig(
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    info_interval=INFO_INTERVAL,
    device=DEVICE
)
train(model, tr_dataset, val_dataset, training_config)
model.to('cpu')
save_model(model, model.config, os.path.join(models_dir_path, model_name + ".pt"))

Using device: cuda
Train dataset length: 891 | Val dataset length: 97
Epoch 0 | Train loss: 10.3739 | Val loss: 10.3755
Epoch 1 | Train loss: 9.4042 | Val loss: 9.1917
Epoch 2 | Train loss: 9.1189 | Val loss: 9.1746
Epoch 3 | Train loss: 9.0712 | Val loss: 9.1867
Epoch 4 | Train loss: 9.0161 | Val loss: 9.2056
Epoch 5 | Train loss: 8.9267 | Val loss: 9.2928
Epoch 6 | Train loss: 8.7923 | Val loss: 9.4011
Epoch 7 | Train loss: 8.6260 | Val loss: 9.5273
Epoch 8 | Train loss: 8.4205 | Val loss: 9.6933
Epoch 9 | Train loss: 8.1410 | Val loss: 9.7382
Epoch 10 | Train loss: 7.7954 | Val loss: 9.7678
Epoch 11 | Train loss: 7.4736 | Val loss: 9.8109
Epoch 12 | Train loss: 7.1664 | Val loss: 10.0548
Epoch 13 | Train loss: 6.8325 | Val loss: 10.0060
Epoch 14 | Train loss: 6.4836 | Val loss: 10.1811
Epoch 15 | Train loss: 6.1207 | Val loss: 10.4241
[save_model] Model saved to models/mickiewicz_gpt.pt


# Further training

In [9]:
NUM_EPOCHS = 50
INFO_INTERVAL = 5
training_config = TrainingConfig(
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    info_interval=INFO_INTERVAL,
    device=DEVICE
)
train(model, tr_dataset, val_dataset, training_config)
save_model(model, model.config, os.path.join(models_dir_path, model_name + "_2.pt"))

Using device: cuda
Train dataset length: 891 | Val dataset length: 97
Epoch 0 | Train loss: 5.5306 | Val loss: 10.4241
Epoch 1 | Train loss: 5.8024 | Val loss: 10.6481
Epoch 6 | Train loss: 4.2039 | Val loss: 11.7748
Epoch 11 | Train loss: 2.6839 | Val loss: 13.3715
Epoch 16 | Train loss: 1.2835 | Val loss: 15.4934
Epoch 21 | Train loss: 0.4543 | Val loss: 17.4838
Epoch 26 | Train loss: 0.1657 | Val loss: 18.7610
Epoch 31 | Train loss: 0.0899 | Val loss: 19.7711
Epoch 36 | Train loss: 0.0576 | Val loss: 20.3308
Epoch 41 | Train loss: 0.0479 | Val loss: 20.8541
Epoch 46 | Train loss: 0.0439 | Val loss: 21.3393
[save_model] Model saved to models/mickiewicz_gpt_2.pt


# Pretraining

In [ ]:
decode(model.generate(encode("Tako Hrabia sługom swym powiadał:"), temperature=0.7))